In [11]:
import os
import sys
import yaml
import pandas as pd
from pathlib import Path

root_path = Path.cwd().parents[1]
os.chdir(root_path)
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.utils.config import load_config

config = load_config()

In [16]:
#read config paths bronze_data:
df = pd.read_csv(config["paths"]["bronze_data"])
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Exploratory Data Analysis
This notebook performs a complete EDA on the customer churn dataset loaded from the configured bronze path. The analysis includes data quality checks, feature summaries, churn distribution, and categorical feature relationships.

In [17]:
# 1. Basic dataset overview
print('Dataset shape:', df.shape)
print('\nColumn data types:')
print(df.dtypes)
print('\nMissing values by column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

# Show a few rows to inspect the data
print('\nSample rows:')
print(df.head().to_string(index=False))

Dataset shape: (7043, 21)

Column data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Missing values by column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies  

In [18]:
# 2. Data type cleanup and quality checks
# Convert TotalCharges to numeric; blank strings become NaN
clean_df = df.copy()
clean_df['TotalCharges'] = pd.to_numeric(clean_df['TotalCharges'].str.strip(), errors='coerce')

print('TotalCharges missing after conversion:', clean_df['TotalCharges'].isna().sum())
print('Rows with missing TotalCharges:')
print(clean_df.loc[clean_df['TotalCharges'].isna(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head().to_string(index=False))

# List categorical and numerical columns explicitly
categorical_cols = clean_df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = clean_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('\nCategorical columns:', categorical_cols)
print('Numeric columns:', numeric_cols)

TotalCharges missing after conversion: 11
Rows with missing TotalCharges:
customerID  tenure  MonthlyCharges  TotalCharges Churn
4472-LVYGI       0           52.55           NaN    No
3115-CZMZD       0           20.25           NaN    No
5709-LVOEQ       0           80.85           NaN    No
4367-NUYAO       0           25.75           NaN    No
1371-DWPAZ       0           56.05           NaN    No

Categorical columns: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']
Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


C:\Users\e702746\AppData\Local\Temp\ipykernel_4972\1205363533.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = clean_df.select_dtypes(include=['object']).columns.tolist()


In [19]:
# 3. Numerical summary
numeric_summary = clean_df[numeric_cols].describe().T
numeric_summary['missing'] = clean_df[numeric_cols].isna().sum()
print(numeric_summary)

                 count         mean          std    min     25%       50%  \
SeniorCitizen   7043.0     0.162147     0.368612   0.00    0.00     0.000   
tenure          7043.0    32.371149    24.559481   0.00    9.00    29.000   
MonthlyCharges  7043.0    64.761692    30.090047  18.25   35.50    70.350   
TotalCharges    7032.0  2283.300441  2266.771362  18.80  401.45  1397.475   

                      75%      max  missing  
SeniorCitizen      0.0000     1.00        0  
tenure            55.0000    72.00        0  
MonthlyCharges    89.8500   118.75        0  
TotalCharges    3794.7375  8684.80       11  


In [20]:
# 4. Categorical summary for feature distributions
for col in categorical_cols:
    print(f'\n=== {col} ===')
    counts = clean_df[col].value_counts(dropna=False)
    print(counts)
    print('Percentages:')
    print((counts / len(clean_df) * 100).round(2))


=== customerID ===
customerID
7590-VHVEG    1
5575-GNVDE    1
3668-QPYBK    1
7795-CFOCW    1
9237-HQITU    1
             ..
6840-RESVB    1
2234-XADUH    1
4801-JZAZL    1
8361-LTMKD    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64
Percentages:
customerID
7590-VHVEG    0.01
5575-GNVDE    0.01
3668-QPYBK    0.01
7795-CFOCW    0.01
9237-HQITU    0.01
              ... 
6840-RESVB    0.01
2234-XADUH    0.01
4801-JZAZL    0.01
8361-LTMKD    0.01
3186-AJIEK    0.01
Name: count, Length: 7043, dtype: float64

=== gender ===
gender
Male      3555
Female    3488
Name: count, dtype: int64
Percentages:
gender
Male      50.48
Female    49.52
Name: count, dtype: float64

=== Partner ===
Partner
No     3641
Yes    3402
Name: count, dtype: int64
Percentages:
Partner
No     51.7
Yes    48.3
Name: count, dtype: float64

=== Dependents ===
Dependents
No     4933
Yes    2110
Name: count, dtype: int64
Percentages:
Dependents
No     70.04
Yes    29.96
Name: count, dtype: float64

=== PhoneSe

In [21]:
# 5. Churn analysis
print('Churn counts and percentages:')
churn_counts = clean_df['Churn'].value_counts()
print(churn_counts)
print((churn_counts / len(clean_df) * 100).round(2))

# Calculate churn rate by key categorical fields
key_features = ['Contract', 'InternetService', 'PaymentMethod', 'gender', 'SeniorCitizen']
for feature in key_features:
    print(f'\nChurn rate by {feature}:')
    ct = pd.crosstab(clean_df[feature], clean_df['Churn'], normalize='index') * 100
    print(ct.round(1))

# Create a tenure bucket for additional churn insight
clean_df['tenure_group'] = pd.cut(clean_df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12', '13-24', '25-48', '49-72'])
print('\nChurn rate by tenure group:')
print(pd.crosstab(clean_df['tenure_group'], clean_df['Churn'], normalize='index').round(3) * 100)

Churn counts and percentages:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.46
Yes    26.54
Name: count, dtype: float64

Churn rate by Contract:
Churn             No   Yes
Contract                  
Month-to-month  57.3  42.7
One year        88.7  11.3
Two year        97.2   2.8

Churn rate by InternetService:
Churn              No   Yes
InternetService            
DSL              81.0  19.0
Fiber optic      58.1  41.9
No               92.6   7.4

Churn rate by PaymentMethod:
Churn                        No   Yes
PaymentMethod                        
Bank transfer (automatic)  83.3  16.7
Credit card (automatic)    84.8  15.2
Electronic check           54.7  45.3
Mailed check               80.9  19.1

Churn rate by gender:
Churn     No   Yes
gender            
Female  73.1  26.9
Male    73.8  26.2

Churn rate by SeniorCitizen:
Churn            No   Yes
SeniorCitizen            
0              76.4  23.6
1              58.3  41.7

Churn rate by tenure group:
Ch

# 6. Numeric correlation and churn relationship

In [ ]:

clean_df['ChurnFlag'] = clean_df['Churn'].map({'No': 0, 'Yes': 1})
correlation = clean_df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'ChurnFlag']].corr()
print(correlation)

# Show the most extreme customers by MonthlyCharges and tenure
print('\nTop 10 customers by MonthlyCharges:')
print(clean_df[['customerID', 'MonthlyCharges', 'TotalCharges', 'tenure', 'Churn']].sort_values('MonthlyCharges', ascending=False).head(10).to_string(index=False))
print('\nTop 10 customers by tenure:')
print(clean_df[['customerID', 'MonthlyCharges', 'TotalCharges', 'tenure', 'Churn']].sort_values('tenure', ascending=False).head(10).to_string(index=False))

                  tenure  MonthlyCharges  TotalCharges  SeniorCitizen  \
tenure          1.000000        0.247900      0.825880       0.016567   
MonthlyCharges  0.247900        1.000000      0.651065       0.220173   
TotalCharges    0.825880        0.651065      1.000000       0.102411   
SeniorCitizen   0.016567        0.220173      0.102411       1.000000   
ChurnFlag      -0.352229        0.193356     -0.199484       0.150889   

                ChurnFlag  
tenure          -0.352229  
MonthlyCharges   0.193356  
TotalCharges    -0.199484  
SeniorCitizen    0.150889  
ChurnFlag        1.000000  

Top 10 customers by MonthlyCharges:
customerID  MonthlyCharges  TotalCharges  tenure Churn
7569-NMZYQ          118.75       8672.45      72    No
8984-HPEMB          118.65       8477.60      71    No
5734-EJKXG          118.60       7365.70      61    No
5989-AXPUC          118.60       7990.05      68    No
8199-ZLLSA          118.35       7804.15      67   Yes
9924-JPRMC          118.20

# 7. Visual EDA
The charts below illustrate churn distribution, numeric relationships, and churn behavior across key customer segments.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid', palette='muted')

plt.figure(figsize=(8, 4))
sns.countplot(data=clean_df, x='Churn')
plt.title('Customer Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(data=clean_df, x='MonthlyCharges', hue='Churn', kde=True, element='step', stat='density', common_norm=False)
plt.title('Monthly Charges Distribution by Churn')
plt.xlabel('Monthly Charges')
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(data=clean_df.dropna(subset=['TotalCharges']), x='TotalCharges', hue='Churn', kde=True, element='step', stat='density', common_norm=False)
plt.title('Total Charges Distribution by Churn')
plt.xlabel('Total Charges')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
contracts = pd.crosstab(clean_df['Contract'], clean_df['Churn'], normalize='index') * 100
contracts.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2')
plt.title('Churn Rate by Contract Type')
plt.ylabel('Percentage')
plt.legend(title='Churn')
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(10, 5))
service = pd.crosstab(clean_df['InternetService'], clean_df['Churn'], normalize='index') * 100
service.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2')
plt.title('Churn Rate by Internet Service')
plt.ylabel('Percentage')
plt.legend(title='Churn')
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
tenure_rate = pd.crosstab(clean_df['tenure_group'], clean_df['Churn'], normalize='index') * 100
tenure_rate.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2')
plt.title('Churn Rate by Tenure Group')
plt.ylabel('Percentage')
plt.xlabel('Tenure Group')
plt.legend(title='Churn')
plt.xticks(rotation=0)
plt.show()

## Notes
- `TotalCharges` is cleaned to numeric values and blanks are converted to `NaN`.
- The analysis uses pandas tables, summary statistics, and visual plots powered by matplotlib/seaborn.
- If you want additional feature-specific plots, I can add them next.